In [2]:
import pandas as pd

In [3]:
df = pd.read_csv('reddit_posts_processed_16Mar.csv')

In [4]:
df.head(20)

,Author,Created_DateTime,Score,Selftext,Tokenized_Selftext,Subreddit,Title,New_Token_Selftext,Lemmatized_Text,Merged_Lemma_Token
0,mamboitalianoo,2025-01-31 21:32:39,1,A couple years ago I had a bout of panic attac...,"['A', 'couple', 'years', 'ago', 'I', 'had', 'a...",Anxiety,Panic attack or vertigo?,couple years ago bout panic attacks related cl...,couple year ago bout panic attack relate claus...,couple year ago bout panic attack relate claus...
1,Midori_sho,2025-01-31 21:31:03,1,School has been kicking my ass lately. My midt...,"['School', 'has', 'been', 'kicking', 'my', 'as...",Anxiety,I think I’m actually stupid as hell,school kicking ass lately midterms kicked ass ...,school kicking ass lately midterm kick ass get...,school kicking ass lately midterm kick ass get...
2,No_Entrepreneur_4395,2025-01-31 21:29:09,1,"I've never been diagnosed by a doctor, but I'm...","['I', ""'ve"", 'never', 'been', 'diagnosed', 'by...",Anxiety,Is being medicated better?,never diagnosed doctor pretty sure adhd anxiet...,never diagnose doctor pretty sure adhd anxiety...,never diagnose doctor pretty sure adhd anxiety...
3,Successful-Bat-5287,2025-01-31 21:22:31,1,Long time anxiety sufferer here. I’ve done the...,"['Long', 'time', 'anxiety', 'sufferer', 'here'...",Anxiety,My body is anxious but I’m not?,long time anxiety sufferer done therapies mg e...,long time anxiety sufferer do therapy mg effex...,long time anxiety sufferer do therapy mg effex...
4,ratherdream,2025-01-31 21:07:44,1,I'm so afraid of losing the ones I love and my...,"['I', ""'m"", 'so', 'afraid', 'of', 'losing', 't...",Anxiety,Afraid to loose people,afraid losing ones love insecurities make bett...,afraid lose one love insecurity make well scar...,afraid lose one love insecurity make well scar...
5,gayLiam116111,2025-01-31 21:01:59,2,"Hi, im 15f and since november of last year i h...","['Hi', ',', 'im', '15f', 'and', 'since', 'nove...",Anxiety,How do i stop thinking that im dying constantly?,hi im since november last year many panic atta...,hi I m since november last year many panic att...,hi I m since november last year many panic att...
6,breenotsoswag,2025-01-31 20:56:06,2,i did it! there are mixed reviews about this p...,"['i', 'did', 'it', '!', 'there', 'are', 'mixed...",Anxiety,i made my psychiatry appointment,mixed reviews psychiatrist butttt glad going n...,mixed review psychiatrist butttt glad go nervo...,mixed review psychiatrist butttt glad go nervo...
7,okokayOKokayk,2025-01-31 20:51:02,4,I got an assessment recently and was diagnosed...,"['I', 'got', 'an', 'assessment', 'recently', '...",Anxiety,Can Low Motivation be Caused by Anxiety?,got assessment recently diagnosed gad however ...,get assessment recently diagnose gad however a...,get assessment recently diagnose gad however a...
8,ssweetdecomposition,2025-01-31 20:49:22,1,I have the most debilitating anxiety in the wo...,"['I', 'have', 'the', 'most', 'debilitating', '...",Anxiety,help,debilitating anxiety world take pills bc scare...,debilitate anxiety world take pill bc scare fu...,debilitate anxiety world take pill bc scare fu...
9,succeedathumanity,2025-01-31 20:28:32,1,"I am an assistant manager of a restaurant, an...","['I', 'am', 'an', 'assistant', 'manager', 'of'...",Anxiety,My General manager cries at work weekly.,assistant manager restaurant general manager c...,assistant manager restaurant general manager c...,assistant manager restaurant general manager c...


## Model

In [5]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [6]:
# Fill missing values in the Selftext column with an empty string
df["Selftext"] = df["Selftext"].fillna("")

In [7]:
# Function to preprocess text
def preprocess_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove special characters
    words = text.split()  # Tokenization
    return " ".join(words)  # Return cleaned text

In [8]:
# Apply text preprocessing
df["Processed_Text"] = df["Selftext"].apply(preprocess_text)

In [9]:
# Use the length of the processed text as a proxy for severity
df["Selftext_Length"] = df["Selftext"].apply(len)

In [10]:
# Fill any NaN values in the target column
df["Selftext_Length"] = df["Selftext_Length"].fillna(0)

In [11]:
# Train/Test Split for ML Model
X_train, X_test, y_train, y_test = train_test_split(df["Processed_Text"], df["Selftext_Length"], test_size=0.3, random_state=42)

In [12]:
# Ensure no NaN values in y_train
y_train = y_train.fillna(0)

In [13]:
# Create TF-IDF + Logistic Regression pipeline
model = make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), LogisticRegression(max_iter=200))

In [14]:
# Train model on predicting severity score (using selftext length as a proxy)
model.fit(X_train, y_train)

: 

In [ ]:
# Predict severity scores for all data
df["Predicted Severity Score"] = model.predict(df["Processed_Text"])

In [ ]:
# Normalize severity scores to range [1, 10]
df["Predicted Severity Score"] = df["Predicted Severity Score"].apply(lambda x: max(1, min(10, round(x / df["Predicted Severity Score"].max() * 10))))

In [ ]:
# Save results
output_csv_path = "severity_test_with_predicted_severity.csv"
df.to_csv(output_csv_path, index=False)

print(f"Processed file saved as {output_csv_path}")

In [ ]:
df_new = pd.read_csv('/content/severity_test_with_nlp_predictions.csv')

In [ ]:
df_new = df_new.drop('Assigned Disorder',axis=1)

In [ ]:
df_new[df_new['Subreddit'] == 'depression'].head(10)

In [ ]:
df_new[df_new['Subreddit'] == 'ADHD'].head(10)